In [5]:
import os
import subprocess
import pycolmap

# === INPUT SETTINGS ===
image_path = "Datasets/Water_Bottle/"
output_path = "Colmap_Output/Water_Bottle_4"
database_path = os.path.join(output_path, "database.db")
sparse_path = os.path.join(output_path, "sparse", "0")
dense_path = os.path.join(output_path, "dense")

# === DENSE QUALITY SETTINGS ===
quality = "high"  # options: "low", "medium", "high"
downscale_map = {
    "low": "4",
    "medium": "2",
    "high": "1"
}

# === CREATE REQUIRED FOLDERS ===
os.makedirs(output_path, exist_ok=True)
os.makedirs(os.path.dirname(sparse_path), exist_ok=True)
os.makedirs(dense_path, exist_ok=True)

# === 1. FEATURE EXTRACTION ===
subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", database_path,
    "--image_path", image_path,
    "--ImageReader.single_camera", "1",
    "--ImageReader.camera_model", "PINHOLE"
], check=True)

# === 2. FEATURE MATCHING ===
subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", database_path
], check=True)

# === 3. SPARSE RECONSTRUCTION ===
print("Running sparse reconstruction...")
reconstructions = pycolmap.incremental_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=sparse_path
)

if not reconstructions:
    raise RuntimeError("Sparse reconstruction failed — no cameras registered.")

print(f"Sparse reconstruction complete. Saved to: {sparse_path}")

# === 4. CHECK FOR REQUIRED SPARSE FILES ===
required_files = ["cameras.bin", "images.bin", "points3D.bin"]
for fname in required_files:
    if not os.path.exists(os.path.join(sparse_path, fname)):
        raise FileNotFoundError(f"Missing {fname} in sparse output.")

# === 5. IMAGE UNDISTORTION ===
print("Undistorting images...")
subprocess.run([
    "colmap", "image_undistorter",
    "--image_path", image_path,
    "--input_path", sparse_path,
    "--output_path", dense_path,
    "--output_type", "COLMAP",
    "--max_image_size", "2000"
], check=True)

# === 6. VERIFY UNDISTORTED OUTPUT ===
if not os.path.exists(os.path.join(dense_path, "images")):
    raise RuntimeError("Undistorted images not found — image_undistorter may have failed.")

# === 7. PATCHMATCH STEREO (DENSE RECONSTRUCTION) ===
print(f"🔍 Running PatchMatch stereo at {quality} quality...")
subprocess.run([
    "colmap", "patch_match_stereo",
    "--workspace_path", dense_path,
    "--workspace_format", "COLMAP",
    "--PatchMatchStereo.geom_consistency", "true",
    "--PatchMatchStereo.depth_map_downscale", downscale_map[quality]
], check=True)

# === 8. VERIFY DEPTH MAPS ===
depth_map_dir = os.path.join(dense_path, "stereo", "depth_maps")
if not os.path.exists(depth_map_dir) or len(os.listdir(depth_map_dir)) == 0:
    raise RuntimeError("No depth maps generated — PatchMatch stereo may have failed.")

# === 9. STEREO FUSION ===
print("Fusing stereo depth maps into dense point cloud...")
fused_ply_path = os.path.join(dense_path, "fused.ply")
subprocess.run([
    "colmap", "stereo_fusion",
    "--workspace_path", dense_path,
    "--workspace_format", "COLMAP",
    "--input_type", "geometric",
    "--output_path", fused_ply_path
], check=True)

# === DONE ===
print(f"Dense reconstruction complete. Point cloud saved to: {fused_ply_path}")


I20250619 11:45:23.850879 48638 misc.cc:44] 
Feature extraction
I20250619 11:45:23.852711 48659 sift.cc:721] Creating SIFT GPU feature extractor
I20250619 11:45:23.853420 48660 feature_extraction.cc:259] Processed file [1/126]
I20250619 11:45:23.853451 48660 feature_extraction.cc:262]   Name:            20250618_133939.jpg
E20250619 11:45:23.853461 48660 feature_extraction.cc:266] 20250618_133939.jpg IMAGE_EXISTS: Features for image were already extracted.
I20250619 11:45:23.853470 48660 feature_extraction.cc:259] Processed file [2/126]
I20250619 11:45:23.853477 48660 feature_extraction.cc:262]   Name:            20250618_133949.jpg
E20250619 11:45:23.853484 48660 feature_extraction.cc:266] 20250618_133949.jpg IMAGE_EXISTS: Features for image were already extracted.
I20250619 11:45:23.853494 48660 feature_extraction.cc:259] Processed file [3/126]
I20250619 11:45:23.853502 48660 feature_extraction.cc:262]   Name:            20250618_133957.jpg
E20250619 11:45:23.853508 48660 feature_ext

🚀 Running sparse reconstruction...


I20250619 11:45:24.283623 137827221075264 database_cache.cc:153]  126 in 0.026s (connected 126)
I20250619 11:45:24.283681 137827221075264 database_cache.cc:164] Loading pose priors...
I20250619 11:45:24.284052 137827221075264 database_cache.cc:175]  0 in 0.000s
I20250619 11:45:24.284063 137827221075264 database_cache.cc:184] Building correspondence graph...
I20250619 11:45:24.515963 137827221075264 database_cache.cc:210]  in 0.232s (ignored 0)
I20250619 11:45:24.516419 137827221075264 timer.cc:91] Elapsed time: 0.005 [minutes]
I20250619 11:45:24.525566 137827221075264 incremental_pipeline.cc:282] Finding good initial image pair
I20250619 11:45:24.867188 137827221075264 incremental_pipeline.cc:306] Initializing with image pair #86 and #40
I20250619 11:45:24.872012 137827221075264 incremental_pipeline.cc:311] Global bundle adjustment
I20250619 11:45:25.209070 137827221075264 incremental_pipeline.cc:390] Registering image #87 (3)
I20250619 11:45:25.209090 137827221075264 incremental_pipel

✅ Sparse reconstruction complete. Saved to: Colmap_Output/Water_Bottle_4/sparse/0


I20250619 11:50:46.862383 137827221075264 timer.cc:91] Elapsed time: 5.377 [minutes]


FileNotFoundError: ❌ Missing cameras.bin in sparse output.